# Projet de groupe Web scraping : véhicules d'occasion (Ottawa-Gatineau)

**Cours :** Collecte et nettoyage des données (projet UA2)
**Groupe :** Yacine Lezoul (2748526) et Mouncif Bouchachi (2743259)
**Activité :** concessionnaires de véhicules d'occasion de la région Ottawa-Gatineau

Ce notebook suit la démarche du document « Projet de groupe Web scraping ». Le code est rangé
dans des modules de `src/` ; le notebook les appelle et affiche les résultats avec les explications.

| Partie du cours | Contenu | Section | Module |
|---|---|---|---|
| 0 | Préparation (environnement, User-Agent, délais) | 0 | `config.py` |
| 1 | Cadre légal et éthique (5 questions, robots.txt, conditions, API) | 1 | `config.py`, `diagnostic.py` |
| 2 | Diagnostic de chaque page (cas A à I) | 2 | `diagnostic.py` |
| 3 et 4 | Collecte et pipeline | 3 | `extraction.py`, `pipeline.py` |
| 5 | Nettoyage, CSV final, journal, paragraphe par URL | 4 et 5 | à venir |

**Organisation des dossiers**

```
Projet_UA2_ETL/
├── notebook/
│   └── projet_vehicules.ipynb   # ce notebook : explications + appels aux modules
├── src/
│   ├── config.py                # réglages : chemins, User-Agent, délais, sites, cas confirmés
│   ├── diagnostic.py            # parties 1 et 2 : URL sans doublons, diagnostic, API, journal
│   ├── extraction.py            # partie 3 : fonction unique d'extraction, pagination, CSV par site
│   ├── pipeline.py              # tout le projet en une commande : python src/pipeline.py
│   ├── projet_scraping.py       # boîte à outils fournie par le cours
│   ├── scraper.py               # pipeline fourni par le cours (référence)
│   └── spider_livres.py         # exemple Scrapy fourni par le cours (référence)
├── pages_brutes/                # HTML téléchargé au diagnostic (non publié sur GitHub)
├── resultats/
│   ├── brut/                    # un CSV par site et par date de collecte
│   ├── vehicules_brut_AAAA-MM-JJ.csv   # tous les véhicules d'une collecte
│   └── journal_tests.csv        # journal de tests (annexe C)
├── requirements.txt             # versions des bibliothèques
└── .gitignore
```

**Exécution.** Dans VS Code, choisir le noyau « Python (projet-scraping) », puis **Run All**
(bouton « Exécuter tout » en haut du notebook). Le même travail peut être lancé sans notebook :
`python src/pipeline.py` depuis la racine du projet.

## 0. Préparation (partie 0 du cours)

### 0.1 Chemins et importation des modules

La cellule retrouve la racine du projet (le dossier qui contient `src/`), quel que soit le dossier
depuis lequel le notebook est lancé, puis importe la boîte à outils du cours et nos modules.

In [1]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd


def trouver_racine(depart=None):
    """Remonte les dossiers jusqu'à trouver celui qui contient src/projet_scraping.py."""
    dossier = Path(depart or Path.cwd()).resolve()
    for candidat in [dossier, *dossier.parents]:
        if (candidat / "src" / "projet_scraping.py").exists():
            return candidat
    raise FileNotFoundError("Dossier src/ introuvable : lancez le notebook depuis le projet.")


RACINE = trouver_racine()
if str(RACINE / "src") not in sys.path:
    sys.path.insert(0, str(RACINE / "src"))

import projet_scraping as ws      # boîte à outils du cours
import config                     # nos réglages
import diagnostic                 # parties 1 et 2
import extraction                 # partie 3

pd.set_option("display.max_colwidth", 120)
print("Racine du projet :", config.RACINE)

Racine du projet : C:\Users\Admin\Desktop\Cours S1\S3\ETL\Projet UA2\Projet_UA2_ETL


### 0.2 Version de Python

Le document « Avant de scraper » (section 3.1) précise que `urllib.robotparser` applique
correctement robots.txt seulement à partir de **Python 3.13.14** ou **3.14.5**. Avec une version
plus ancienne, une page interdite peut être déclarée permise : on vérifie avant toute collecte.

In [2]:
v = sys.version_info
version_ok = (v[:2] == (3, 13) and v >= (3, 13, 14)) or (v[:2] == (3, 14) and v >= (3, 14, 5)) or v[:2] > (3, 14)
print(f"Python {v.major}.{v.minor}.{v.micro} :", "conforme" if version_ok else "TROP ANCIENNE, mettre Python à jour")
assert version_ok, "Mettre Python à jour avant de continuer (voir « Avant de scraper », section 3.1)."

Python 3.14.6 : conforme


### 0.3 Réglages utilisés partout (`src/config.py`)

| Réglage | Valeur | Pourquoi |
|---|---|---|
| `USER_AGENT` | nom du projet + nos deux courriels | Nous identifier honnêtement : un administrateur peut nous contacter au lieu de nous bloquer |
| `DELAI_MINIMUM` | 2 s | Le cours exige au moins 1 s entre deux requêtes au même site |
| Crawl-delay | lu dans robots.txt (5 s pour les sites D2C Media) | On applique toujours le plus grand des délais |
| `MAX_PAGES_PAR_SITE` | 20 | Garde-fou : jamais plus de 20 pages de liste par site |
| `TIMEOUT` | 15 s (valeur de `projet_scraping.py`) | Abandonner une requête qui ne répond pas |

In [3]:
session = ws.creer_session(config.USER_AGENT)
print("User-Agent :", session.headers["User-Agent"])
print("Délai minimum :", config.DELAI_MINIMUM, "s ; pages max par site :", config.MAX_PAGES_PAR_SITE,
      "; timeout :", ws.TIMEOUT, "s")

User-Agent : CoursCollecteDonnees-LaCite/1.0 (projet etudiant; contact: 2748526@collegelacite.ca, 2743259@collegelacite.ca)
Délai minimum : 2.0 s ; pages max par site : 20 ; timeout : 15 s


## 1. Cadre légal et éthique (partie 1 du cours)

### 1.1 Les cinq questions avant toute collecte

| Question du cours | Notre réponse |
|---|---|
| 1. Existe-t-il une API officielle ou un jeu de données ouvert ? | Non. Aucun lien « API » ou « données » sur les sites, aucun jeu de données d'inventaire de concessionnaires sur ouvert.canada.ca ni sur donneesquebec.ca (Statistique Canada publie seulement des ventes agrégées). Il existe des fournisseurs commerciaux payants (MarketCheck, VinAudit), ni officiels ni ouverts. Vérification automatique complémentaire à la section 2. |
| 2. Que dit robots.txt ? | Lu pour chaque site (tableau 1.2), puis revérifié automatiquement par `robots_autorise()` avant chaque requête. |
| 3. Que disent les conditions d'utilisation ? | Lues pour chaque site. Les sites qui interdisent les robots ou le moissonnage sont exclus (section 1.4). |
| 4. Y a-t-il des renseignements personnels ? | Non : un inventaire décrit des produits (prix, année, kilométrage). On ne collecte ni noms ni téléphones. |
| 5. Mon script est-il poli ? | User-Agent identifiable, délai d'au moins 2 s (5 s si robots.txt le demande), pages de liste seulement, arrêt immédiat en cas de blocage. |

**Contexte canadien.** Les sites de Gatineau relèvent de la Loi 25 (Québec) et ceux d'Ottawa de la
LPRPDE : aucune difficulté ici puisqu'aucun renseignement personnel n'est collecté. Pour le droit
d'auteur, le cours rappelle que les faits bruts (un prix, une date) ne sont pas protégés : nous
collectons des faits, ni photos ni textes rédigés. **Usage strictement pédagogique** : les données ne
seront ni publiées ni exploitées en dehors de ce travail.

> **À refaire avant la remise :** revérifier la question 1 et mettre à jour la date dans `config.py`.

In [4]:
print(config.VERIFICATION_API_MANUELLE)

Aucune API officielle ni donnée ouverte trouvée (pied de page du site, ouvert.canada.ca, donneesquebec.ca ; vérifié le 2026-10-09). Seules des sources commerciales payantes existent.


### 1.2 Les 18 sites validés

Pour chaque site : sa plateforme (les sites d'une même plateforme ont le même HTML), son
Crawl-delay relevé dans robots.txt, et les règles relevées dans robots.txt et dans les conditions
d'utilisation (vérifiées le 9 octobre 2026). La colonne `collecter` vaut `False` pour un site noté
dans le journal mais jamais collecté.

In [5]:
pd.DataFrame(config.SITES)[["nom", "ville", "plateforme", "delai_robots", "collecter", "regles"]]

,nom,ville,plateforme,delai_robots,collecter,regles
0,Hyundai Gatineau,Gatineau,Magnetis,NaN,True,robots.txt : rien n'est bloqué. Conditions : usage personnel et non commercial.
1,Gatineau Acura,Gatineau,Magnetis,NaN,True,robots.txt : rien n'est bloqué. Conditions : usage personnel et non commercial.
2,Gatineau Kia,Gatineau,Magnetis,NaN,True,robots.txt : rien n'est bloqué. Conditions : usage personnel et non commercial.
3,Dupont & Dupont Honda,Gatineau,D2C Media,5.0,True,robots.txt : Crawl-delay 5 s ; /used/detail.html et /json/ interdits (liste seulement). Conditions : usage personnel...
4,Hull Hyundai,Gatineau (Hull),D2C Media,5.0,True,robots.txt : Crawl-delay 5 s ; /used/detail.html et /json/ interdits (liste seulement). Conditions : usage personnel...
5,Hull Nissan,Gatineau (Hull),D2C Media,5.0,True,robots.txt : Crawl-delay 5 s ; /used/detail.html et /json/ interdits (liste seulement). Conditions : usage personnel...
6,Buckingham Chevrolet Buick GMC,Gatineau (Buckingham),D2C Media,5.0,True,robots.txt : Crawl-delay 5 s ; /used/detail.html et /json/ interdits (liste seulement). Conditions : usage personnel...
7,Ottawa Honda,Ottawa,EDealer,NaN,False,robots.txt : /inventory/ permis ; URL avec filtres interdites sauf ?condition=USED. Aucune condition d'utilisation t...
8,Bank Street Mazda,Ottawa,Dilawri,NaN,True,robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée.
9,Bank Street Hyundai,Ottawa,Dilawri,NaN,True,robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée.


### 1.3 Règle du groupe : zéro risque éthique

**En cas de doute sur la légalité ou l'éthique d'une collecte, on ne collecte pas.** Tout ce qui
est interdit, ou dont on ne peut pas être sûrs que c'est permis, est évité et noté dans le journal.

| Niveau | Sites ou pratiques | Raison |
|---|---|---|
| 🔴 **Interdit, on n'y touche pas** | Toyota Gatineau, Villa Toyota | Conditions SM360 : robots interdits |
| | Bel-Air Toyota, Car Canada, groupe Myers (16 concessionnaires) | Conditions : « spider, crawl, or scrape » interdits |
| | Ottawa Honda | Protection Cloudflare (403) : la contourner est interdit |
| | Pages détail et chemin `/json/` des sites D2C Media | Interdits par robots.txt |
| 🟠 **Évité par prudence** | Fichier `inventory-index.json` de Buckingham GM (cas E1b) | Fichier interne du chatbot : pas interdit mot à mot par robots.txt, mais on ne peut pas être sûrs que son usage par un robot est voulu |
| | Playwright pour charger le reste de l'inventaire D2C (cas E2) | Reviendrait à imiter un navigateur pour obtenir ce que le site ne sert pas aux robots |
| | Véhicules D2C absents du HTML (Buckingham 25, Hull Hyundai environ 9, Dupont 2) | Seule voie d'accès : JavaScript ou chemins interdits |
| | Téléphones, noms de vendeurs, descriptions rédigées, photos | Renseignements de contact et contenu protégé : on ne garde que des faits bruts |
| 🟢 **Permis** | HTML des listes d'inventaire des 17 sites accessibles | robots.txt le permet, aucune condition n'interdit les robots |
| | JSON-LD intégré dans ces pages (cas E1a) | Fait partie de la page déjà téléchargée, aucune requête de plus |
| | Pages suivantes par `?vip=` et `?pindex=` | Liens présents dans le HTML, non bloqués par robots.txt |

**Réserve.** Les sites Magnetis (3) et D2C Media (4) limitent l'usage du contenu à un « usage
personnel et non commercial ». Notre usage est pédagogique, non commercial et non publié, et ces
sites ont été validés par le professeur.

### 1.4 Sites écartés avant toute collecte

Ces sites ont été examinés puis écartés parce que leurs conditions d'utilisation interdisent la
collecte automatisée. Conformément aux consignes, ils sont notés mais ne comptent pas parmi nos
sites. Aucune requête automatisée ne leur est envoyée.

In [6]:
pd.DataFrame(config.SITES_EXCLUS)

,site,url,motif
0,Toyota Gatineau,https://www.toyotagatineau.ca/fr/conditions-utilisation,"Conditions (plateforme SM360) : interdit « d'utiliser un robot, ou tout autre appareil, processus ou moyen automatiq..."
1,Villa Toyota,https://www.villatoyota.com/fr/conditions-utilisation,Mêmes conditions SM360 : robots interdits.
2,Bel-Air Toyota,https://www.belairtoyota.ca/terms-conditions,"Conditions, section 12 : interdit de « spider, crawl, or scrape »."
3,Car Canada,https://www.carcanada.com/terms-conditions,"Conditions, section 12 : interdit de « spider, crawl, or scrape »."
4,Groupe Myers (16 concessionnaires),https://www.myers.ca/terms-conditions,"Conditions, section 12 : interdit de « spider, crawl, or scrape »."


### 1.5 Liste des URL à traiter, sans doublons

Avant toute requête, `diagnostic.sites_sans_doublons()` retire les doublons : deux URL sont
identiques si elles ne diffèrent que par la casse du domaine ou par une barre oblique finale.
La même liste dédoublonnée sert au diagnostic (section 2) et à la collecte (section 3).

In [7]:
SITES_A_TRAITER = diagnostic.sites_sans_doublons(config.SITES)
print(f"{len(config.SITES)} sites déclarés, {len(SITES_A_TRAITER)} URL uniques à traiter.")

18 sites déclarés, 18 URL uniques à traiter.


## 2. Diagnostic (partie 2 du cours)

Pour chaque URL, `diagnostic.diagnostiquer_site()` applique l'arbre de décision du cours :

1. `ws.diagnostiquer(url)` lit robots.txt, envoie **une** requête et observe la page : statut
   HTTP, protection anti-robot, encodage, JavaScript, JSON intégré, tableaux, pagination, blocs
   répétés. Il en déduit le ou les cas (A à I).
2. Si le cas est **STOP** (robots.txt) ou **G** (blocage), on s'arrête : on ne contourne jamais.
3. Sinon, la page est téléchargée une fois avec `ws.obtenir()` (qui revérifie robots.txt, gère
   les erreurs 429 et 5xx, et s'arrête sur un 403) et enregistrée dans `pages_brutes/`.
4. `diagnostic.chercher_api()` cherche les signes d'**API** ou de données structurées (question 1
   de la partie 1, cas E) : JSON-LD, scripts `application/json`, adresses d'API dans le code, et
   permission robots.txt pour les chemins `/api/` et `/json/`.

Environ 3 minutes : 2 requêtes par site, avec le délai entre chaque.

In [8]:
lignes_diagnostic = diagnostic.diagnostiquer_tous(SITES_A_TRAITER, session)
journal = diagnostic.construire_journal(lignes_diagnostic)


Hyundai Gatineau (Gatineau, Magnetis)
──────────────────────────────────────────────────────────────────────
DIAGNOSTIC : https://hyundaigatineau.ca/en/pre-owned
──────────────────────────────────────────────────────────────────────
  robots.txt       : autorisé par robots.txt
  Statut HTTP      : 200
  Type de contenu  : text/html
  Encodage         : déclaré=UTF-8  requests=UTF-8  détecté=utf-8
  Taille           : 1147.7 Ko en 0.48 s
  Serveur          : LiteSpeed
  Titre            : Used and Certified Pre-Owned vehicles | Hyundai of Gatineau
  Texte visible    : 26025 caractères, 90 scripts
  Tableaux / liens : 0 / 358
  Blocs répétés    : div.mb-4.vehicle-card-wrapper x20
  Pagination       : https://hyundaigatineau.ca/en/pre-owned?vip=2
  Connexion        : non
  Signes JS        : aucun
  JSON intégré     : JSON-LD (schema.org), script application/json, variable JavaScript contenant des données
  Protection       : aucune détectée

  CAS DÉTECTÉS   : C — pagination | D — liste

19:50:19 | INFO    | 200  https://hyundaigatineau.ca/en/pre-owned  (0.43s)


→ HTML enregistré : pages_brutes\hyundai_gatineau.html (1148 Ko)
→ JSON-LD : WebPage x1, WebSite x1, AutoDealer/LocalBusiness x1, BreadcrumbList x1, ItemList(20) x1 ; véhicules décrits : 20
→ robots.txt pour les API : /api/ permis ; /json/ permis

Gatineau Acura (Gatineau, Magnetis)
──────────────────────────────────────────────────────────────────────
DIAGNOSTIC : https://gatineauacura.com/en/pre-owned
──────────────────────────────────────────────────────────────────────
  robots.txt       : autorisé par robots.txt
  Statut HTTP      : 200
  Type de contenu  : text/html
  Encodage         : déclaré=UTF-8  requests=UTF-8  détecté=utf-8
  Taille           : 1580.9 Ko en 0.6 s
  Serveur          : LiteSpeed
  Titre            : Used and Certified Pre-Owned vehicles | Gatineau Acura
  Texte visible    : 35728 caractères, 89 scripts
  Tableaux / liens : 0 / 437
  Blocs répétés    : div.mb-4.vehicle-card-wrapper x29
  Pagination       : non
  Connexion        : non
  Signes JS        : auc

19:50:27 | INFO    | 200  https://gatineauacura.com/en/pre-owned  (0.50s)


→ HTML enregistré : pages_brutes\gatineau_acura.html (1581 Ko)
→ JSON-LD : WebPage x1, WebSite x1, AutoDealer/LocalBusiness x1, BreadcrumbList x1, ItemList(29) x1 ; véhicules décrits : 29
→ robots.txt pour les API : /api/ permis ; /json/ permis

Gatineau Kia (Gatineau, Magnetis)
──────────────────────────────────────────────────────────────────────
DIAGNOSTIC : https://www.gatineaukia.ca/en/pre-owned
──────────────────────────────────────────────────────────────────────
  robots.txt       : autorisé par robots.txt
  Statut HTTP      : 200
  Type de contenu  : text/html
  Encodage         : déclaré=UTF-8  requests=UTF-8  détecté=utf-8
  Taille           : 1024.8 Ko en 0.99 s
  Serveur          : LiteSpeed
  Titre            : Pre-Owned Kia Vehicles & Other Brands | Gatineau Kia
  Texte visible    : 21558 caractères, 81 scripts
  Tableaux / liens : 0 / 302
  Blocs répétés    : div.mb-4.vehicle-card-wrapper x20
  Pagination       : https://www.gatineaukia.ca/en/pre-owned?vip=2
  Connexion

...

→ HTML enregistré : pages_brutes\orleans_mitsubishi.html (328 Ko)
→ JSON-LD : aucun ; véhicules décrits : 0
→ robots.txt pour les API : /api/ permis ; /json/ permis

Hyundai On Hunt Club (Ottawa (Nepean), Dilawri)
──────────────────────────────────────────────────────────────────────
DIAGNOSTIC : https://www.hyundaionhuntclub.com/hyundai/used~vehicles/inventory.html?type=U
──────────────────────────────────────────────────────────────────────
  robots.txt       : autorisé par robots.txt
  Statut HTTP      : 200
  Type de contenu  : text/html
  Encodage         : déclaré=utf-8  requests=utf-8  détecté=utf-8
  Taille           : 461.9 Ko en 0.16 s
  Serveur          : cloudflare
  Titre            : Used Inventory Search at Hyundai On Hunt Club - Find Your Pre-Owned Vehicle at Your Local Ottawa Used Car Dealer
  Texte visible    : 15129 caractères, 21 scripts
  Tableaux / liens : 0 / 273
  Blocs répétés    : div.chrome-payment-result.vsearchresult x12
  Pagination       : non
  Connexion

19:52:22 | INFO    | 200  https://www.hyundaionhuntclub.com/hyundai/used~vehicles/inventory.html?type=U  (0.13s)


→ HTML enregistré : pages_brutes\hyundai_hunt_club.html (462 Ko)
→ JSON-LD : aucun ; véhicules décrits : 0
→ robots.txt pour les API : /api/ permis ; /json/ permis

Kia On Hunt Club (Ottawa (Nepean), Dilawri)
──────────────────────────────────────────────────────────────────────
DIAGNOSTIC : https://www.kiaonhuntclub.com/kia/used-inventory.html?type=U
──────────────────────────────────────────────────────────────────────
  robots.txt       : autorisé par robots.txt
  Statut HTTP      : 200
  Type de contenu  : text/html
  Encodage         : déclaré=utf-8  requests=utf-8  détecté=utf-8
  Taille           : 494.4 Ko en 0.33 s
  Serveur          : cloudflare
  Titre            : Ottawa Kia On Hunt Club's used search for used cars in Ottawa, K2E 1A5 area.
  Texte visible    : 16998 caractères, 23 scripts
  Tableaux / liens : 2 / 253
  Blocs répétés    : div.chrome-payment-result.vsearchresult x12
  Pagination       : non
  Connexion        : non
  Signes JS        : aucun
  JSON intégré   

19:52:28 | INFO    | 200  https://www.kiaonhuntclub.com/kia/used-inventory.html?type=U  (0.14s)


→ HTML enregistré : pages_brutes\kia_hunt_club.html (494 Ko)
→ JSON-LD : JSON-LD illisible x1 ; véhicules décrits : 0
→ robots.txt pour les API : /api/ permis ; /json/ permis


### 2.1 Résumé du diagnostic automatique

In [9]:
journal.reindex(columns=["nom", "plateforme", "statut", "cas_diagnostic", "blocs_repetes",
                         "json_ld_vehicules", "robots_chemins_api"])

,nom,plateforme,statut,cas_diagnostic,blocs_repetes,json_ld_vehicules,robots_chemins_api
0,Hyundai Gatineau,Magnetis,200,C — pagination | D — liste vers pages détail (à vérifier),div.mb-4.vehicle-card-wrapper x20,20.0,/api/ permis ; /json/ permis
1,Gatineau Acura,Magnetis,200,D — liste vers pages détail (à vérifier),div.mb-4.vehicle-card-wrapper x29,29.0,/api/ permis ; /json/ permis
2,Gatineau Kia,Magnetis,200,C — pagination | D — liste vers pages détail (à vérifier),div.mb-4.vehicle-card-wrapper x20,20.0,/api/ permis ; /json/ permis
3,Dupont & Dupont Honda,D2C Media,200,D — liste vers pages détail (à vérifier),li.carBoxWrapper x36,36.0,/api/ permis ; /json/ INTERDIT
4,Hull Hyundai,D2C Media,200,B — tableau(x) HTML | D — liste vers pages détail (à vérifier),li.carBoxWrapper x36,36.0,/api/ permis ; /json/ INTERDIT
5,Hull Nissan,D2C Media,200,D — liste vers pages détail (à vérifier),li.carBoxWrapper x12,12.0,/api/ permis ; /json/ INTERDIT
6,Buckingham Chevrolet Buick GMC,D2C Media,200,D — liste vers pages détail (à vérifier),li.carBoxWrapper x36,36.0,/api/ permis ; /json/ INTERDIT
7,Ottawa Honda,EDealer,403,G — accès bloqué,,NaN,NaN
8,Bank Street Mazda,Dilawri,200,D — liste vers pages détail (à vérifier),div.chrome-payment-result.vsearchresult x12,0.0,/api/ permis ; /json/ permis
9,Bank Street Hyundai,Dilawri,200,D — liste vers pages détail (à vérifier),div.chrome-payment-result.vsearchresult x12,0.0,/api/ permis ; /json/ permis


### 2.2 Du diagnostic automatique au cas confirmé

Le cours demande de confirmer chaque diagnostic en ouvrant la page dans le navigateur (F12).
Vérifications faites le 9 octobre 2026 dans Microsoft Edge :

- **Hyundai Gatineau** : 2 pages (`?vip=2`) ; Ctrl+U montre un JSON-LD `ItemList` de 20 véhicules.
- **Bank Street Mazda** : la liste se charge au défilement dans le navigateur, mais le HTML contient
  des liens `?pindex=` qui donnent les mêmes pages sans JavaScript.
- **Ottawa Honda** : la page s'ouvre lentement dans le navigateur (vérification Cloudflare
  silencieuse), alors que notre script reçoit un 403.
- **Buckingham GM et Hull Hyundai** : capture de l'onglet Réseau (fichier HAR). Aucune requête ne
  charge de véhicules au défilement. Chez Buckingham, la page charge le fichier interne du chatbot
  `inventory-index.json` (61 véhicules d'occasion et démos), **non utilisé par prudence** (section 1.3).

Les autres sites ont la même structure HTML que le site vérifié de leur plateforme ; leur cas
repose sur ce site de référence et sur le HTML enregistré.

**Les cas du cours appliqués à nos sites**

| Cas | Ce que ça veut dire | Sites concernés | Difficulté rencontrée | Ce qu'on fait |
|---|---|---|---|---|
| **A** HTML statique | Les véhicules sont dans le code HTML | Les 17 sites accessibles | Chaque plateforme a ses propres balises | Un lecteur de page par plateforme dans `extraction.py` |
| **B** Tableau HTML | Données dans un `<table>` | Aucun (fausse alerte sur 5 sites Dilawri et Hull Hyundai) | Le diagnostic a pris des tableaux de calcul de paiement pour des données | Cas écarté |
| **C** Pagination | Inventaire réparti sur plusieurs pages | Hyundai Gatineau, Gatineau Kia (`?vip=`), 10 sites Dilawri (`?pindex=`) | Défilement continu dans le navigateur ; liens de page non vus par le diagnostic | On suit les liens de page du HTML, avec délai et arrêt |
| **D** Liste vers pages détail | Une page par véhicule | Possible partout, interdit chez D2C Media | Une requête de plus par véhicule | Non utilisé : la liste suffit, et c'est plus poli |
| **E1a** JSON intégré | Données en JSON dans la page | Magnetis (`ItemList`), D2C Media (`Vehicle`) | Le JSON-LD ne contient pas le kilométrage | JSON-LD + HTML combinés, sans requête de plus |
| **E1b** API cachée | La page appelle un fichier JSON | Buckingham GM (`inventory-index.json`) | Fichier interne du chatbot, autorisation incertaine | Non utilisé par prudence |
| **E2** Navigateur automatisé | Playwright exécute le JavaScript | Aurait donné tout l'inventaire D2C | Revient à imiter un navigateur | Non utilisé |
| **F** Connexion | Compte requis | Aucun | | |
| **G** Blocage | Le site refuse l'accès | Ottawa Honda (403, Cloudflare) | Le site s'ouvre dans un vrai navigateur, pas pour notre script | Arrêt, aucun contournement |
| **H** Encodage | Accents mal décodés | 4 sites D2C Media (iso-8859-1) | Risque de « Ã© » au lieu de « é » | `corriger_encodage()` du cours (appelé par `obtenir()`) |
| **I** Grand volume | Scrapy | Aucun | Environ 60 pages et 900 véhicules | Une boucle requests avec délai suffit |

Les cas confirmés et la technique retenue pour chaque site sont dans `config.CAS_CONFIRMES` et
`config.TECHNIQUES` ; `diagnostic.construire_journal()` les a déjà reportés dans le journal.

In [10]:
journal.reindex(columns=["nom", "cas_diagnostic", "cas_confirme", "technique"])

,nom,cas_diagnostic,cas_confirme,technique
0,Hyundai Gatineau,C — pagination | D — liste vers pages détail (à vérifier),"A + C + E1a (vérifié : 2 pages, lien ?vip=2 ; Ctrl+U : JSON-LD ItemList de 20 véhicules par page ; pages détail non ...",requests + BeautifulSoup + JSON-LD intégré ; pages ?vip= avec délai
1,Gatineau Acura,D — liste vers pages détail (à vérifier),"A + E1a (même plateforme que Hyundai Gatineau ; une seule page, 29 fiches et ItemList de 29 véhicules dans le HTML)",requests + BeautifulSoup + JSON-LD intégré ; pages ?vip= avec délai
2,Gatineau Kia,C — pagination | D — liste vers pages détail (à vérifier),"A + C + E1a (même plateforme que Hyundai Gatineau ; lien ?vip=2, ItemList de 20 véhicules par page)",requests + BeautifulSoup + JSON-LD intégré ; pages ?vip= avec délai
3,Dupont & Dupont Honda,D — liste vers pages détail (à vérifier),A + H (36 fiches dans le HTML sur 38 annoncées ; les 2 autres ne sont accessibles que par JavaScript ou par des chem...,requests + BeautifulSoup + JSON-LD intégré ; correction de l'encodage ; liste seulement
4,Hull Hyundai,B — tableau(x) HTML | D — liste vers pages détail (à vérifier),"A + H (HAR : aucune requête au défilement ; 39 fiches dans le HTML, seules collectées ; pages détail et /json/ inter...",requests + BeautifulSoup + JSON-LD intégré ; correction de l'encodage ; liste seulement
5,Hull Nissan,D — liste vers pages détail (à vérifier),A + H (12 fiches dans le HTML sur 12 annoncées : inventaire complet ; encodage iso-8859-1),requests + BeautifulSoup + JSON-LD intégré ; correction de l'encodage ; liste seulement
6,Buckingham Chevrolet Buick GMC,D — liste vers pages détail (à vérifier),"A + H (HAR : 36 fiches dans le HTML, aucune requête au défilement ; la page charge /js/json/chatboost/inventory/inve...",requests + BeautifulSoup + JSON-LD intégré ; correction de l'encodage ; liste seulement
7,Ottawa Honda,G — accès bloqué,"G (403 + défi Cloudflare « Just a moment » ; dans le navigateur, chargement lent = vérification silencieuse ; ARRÊT,...",Aucune (arrêt volontaire)
8,Bank Street Mazda,D — liste vers pages détail (à vérifier),"A + C (vérifié : défilement continu dans le navigateur, mais liens ?pindex= présents dans le HTML ; on suit ces lien...",requests + BeautifulSoup ; pages ?pindex= avec délai
9,Bank Street Hyundai,D — liste vers pages détail (à vérifier),A + C (même plateforme que Bank Street Mazda ; liens ?pindex= dans le HTML),requests + BeautifulSoup ; pages ?pindex= avec délai


In [11]:
chemin = diagnostic.enregistrer_journal(journal)
print("Journal enregistré :", chemin.relative_to(config.RACINE))

Journal enregistré : resultats\journal_tests.csv


## 3. Collecte (parties 3 et 4 du cours)

### 3.1 Principe

Une **seule fonction**, `extraction.extraire_vehicules(site, session)`, est appelée **une fois par
URL** de la liste dédoublonnée (section 1.5). Pour chaque site, elle :

1. choisit le **lecteur de page** de sa plateforme (`lire_page_magnetis`, `lire_page_d2c`,
   `lire_page_dilawri`) : les sites d'une même plateforme ont le même HTML ;
2. télécharge la page avec `ws.obtenir()` : robots.txt revérifié, délai du site respecté,
   erreurs 429 et 5xx gérées, arrêt sur 403 ;
3. suit la **pagination présente dans le HTML** (`?vip=` ou `?pindex=`), en s'arrêtant dès qu'une
   page n'apporte aucun nouveau véhicule, que le total annoncé est atteint, ou après 20 pages ;
4. ne garde chaque véhicule **qu'une fois** (clé : NIV, sinon identifiant de l'annonce), car
   certaines pages Dilawri répètent des véhicules mis en vedette.

**Règle zéro risque appliquée au code** : pages de liste seulement (jamais les pages détail), seulement
le HTML servi par le site et le JSON-LD qu'il contient, ni API interne, ni navigateur automatisé, et
aucun renseignement personnel. Ottawa Honda (`collecter = False`) n'est jamais contacté.

**Colonnes produites** (une ligne par véhicule)

| Groupe | Colonnes |
|---|---|
| Site | `site_id`, `concessionnaire`, `ville`, `province`, `plateforme`, `date_collecte` |
| Annonce | `url_page`, `url_fiche`, `id_annonce`, `no_stock`, `niv` |
| Véhicule | `annee`, `marque`, `modele`, `version`, `etat` |
| Prix | `prix_affiche`, `rabais`, `frais`, `prix_final` |
| Caractéristiques | `km`, `transmission`, `traction`, `carburant`, `moteur`, `couleur_ext`, `categorie` |

**Prix selon la plateforme** (à harmoniser au nettoyage, section 4) :

| Plateforme | `prix_affiche` | `rabais` | `frais` | `prix_final` |
|---|---|---|---|---|
| Magnetis | « Price » | « Rebate » | absent | « Your price » |
| D2C Media | ancien prix barré (sinon prix actuel) | différence | absent | prix actuel |
| Dilawri | « Price » du détail de prix | réductions | Admin Fee, OMVIC Fee | « Dilawri selling price » |

Durée : environ 5 à 10 minutes (environ 60 pages, 2 à 5 s entre chaque).

In [12]:
vehicules, rapports = extraction.collecter_tous(SITES_A_TRAITER, session)


Hyundai Gatineau (Magnetis)


19:52:31 | INFO    | 200  https://hyundaigatineau.ca/en/pre-owned  (0.52s)


  page 1 : 20 nouveaux véhicules (total 20)


19:52:34 | INFO    | 200  https://hyundaigatineau.ca/en/pre-owned?vip=2  (0.37s)


  page 2 : 14 nouveaux véhicules (total 34)
  → 34 véhicules ; aucune remarque

Gatineau Acura (Magnetis)


...

  page 1 relue dans l'ordre de tri des liens de pagination
  page 1 : 12 nouveaux véhicules (total 24)


19:55:31 | INFO    | 200  https://www.kiaonhuntclub.com/kia/used-inventory.html?type=U&syear=%25&make=%25&model=%25&trim=%25&colour=%25&onsale=%25&onspecial=%25&demo=%25&cpo=%25&minmiles=0&maxmiles=1000000&minprice=0&maxprice=1000000&sort=tadded_asc&psize=12&stknum=%25&b_stock=%25&as_is=%25&text_filter=&body_style=%25&engine=%25&trans=%25&pindex=2  (0.15s)
19:55:33 | INFO    | 200  https://www.kiaonhuntclub.com/kia/used-inventory.html?type=U&syear=%25&make=%25&model=%25&trim=%25&colour=%25&onsale=%25&onspecial=%25&demo=%25&cpo=%25&minmiles=0&maxmiles=1000000&minprice=0&maxprice=1000000&sort=tadded_asc&psize=12&stknum=%25&b_stock=%25&as_is=%25&text_filter=&body_style=%25&engine=%25&trans=%25&pindex=3  (0.16s)


  page 2 : 12 nouveaux véhicules (total 36)


19:55:36 | INFO    | 200  https://www.kiaonhuntclub.com/kia/used-inventory.html?type=U&syear=%25&make=%25&model=%25&trim=%25&colour=%25&onsale=%25&onspecial=%25&demo=%25&cpo=%25&minmiles=0&maxmiles=1000000&minprice=0&maxprice=1000000&sort=tadded_asc&psize=12&stknum=%25&b_stock=%25&as_is=%25&text_filter=&body_style=%25&engine=%25&trans=%25&pindex=4  (0.13s)


  page 3 : 12 nouveaux véhicules (total 48)
  page 4 : 4 nouveaux véhicules (total 52)
  → 52 véhicules ; aucune remarque


### 3.2 Résultat par site

In [13]:
resume_collecte = pd.DataFrame(rapports)
resume_collecte["remarques"] = resume_collecte["remarques"].apply(" ".join)
resume_collecte.reindex(columns=["id", "nb_lignes", "pages_collectees", "fichier_csv", "remarques"])

,id,nb_lignes,pages_collectees,fichier_csv,remarques
0,hyundai_gatineau,34,2,resultats\brut\hyundai_gatineau_2026-10-22.csv,
1,gatineau_acura,29,1,resultats\brut\gatineau_acura_2026-10-22.csv,
2,gatineau_kia,47,3,resultats\brut\gatineau_kia_2026-10-22.csv,
3,dupont_honda,36,1,resultats\brut\dupont_honda_2026-10-22.csv,
4,hull_hyundai,36,1,resultats\brut\hull_hyundai_2026-10-22.csv,
5,hull_nissan,12,1,resultats\brut\hull_nissan_2026-10-22.csv,
6,buckingham_gm,36,1,resultats\brut\buckingham_gm_2026-10-22.csv,
7,ottawa_honda,0,0,NaN,Site non collecté (règle zéro risque : voir le diagnostic).
8,bank_street_mazda,80,7,resultats\brut\bank_street_mazda_2026-10-22.csv,
9,bank_street_hyundai,55,5,resultats\brut\bank_street_hyundai_2026-10-22.csv,


In [14]:
print(f"{len(vehicules)} véhicules collectés sur {vehicules['site_id'].nunique()} sites")
vehicules.groupby(["province", "plateforme"]).size().rename("véhicules").to_frame()

801 véhicules collectés sur 17 sites


véhicules
province plateforme           
ON       Dilawri           571
QC       D2C Media         120
         Magnetis          110

In [15]:
vehicules.head(10)

,site_id,concessionnaire,ville,province,plateforme,date_collecte,url_page,url_fiche,id_annonce,no_stock,...,rabais,frais,prix_final,km,transmission,traction,carburant,moteur,couleur_ext,categorie
0,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2015/hyundai/santa-fe-sport/9556,9556,6851A,...,NaN,NaN,10995.0,87382.0,Automatic,FWD,NaN,NaN,NaN,NaN
1,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2018/hyundai/elantra-gt/9770,9770,6449A,...,NaN,NaN,12995.0,100447.0,5 VITESSES MANUELLES,FWD,NaN,NaN,NaN,NaN
2,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2017/hyundai/tucson/9712,9712,6868B,...,NaN,NaN,13495.0,132639.0,Automatic,AWD,NaN,NaN,NaN,NaN
3,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2019/hyundai/tucson/9128,9128,6267B,...,2100.0,NaN,13895.0,162151.0,Automatic,AWD,NaN,NaN,NaN,NaN
4,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2021/hyundai/tucson/9502,9502,6933A,...,NaN,NaN,16495.0,173693.0,Automatic,FWD,NaN,NaN,NaN,NaN
5,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2019/hyundai/santa-fe/9568,9568,27010A,...,NaN,NaN,16995.0,92462.0,Automatic,NaN,Gasoline,NaN,NaN,NaN
6,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2020/hyundai/santa-fe/9463,9463,6378A,...,700.0,NaN,17995.0,110710.0,Automatic,FWD,NaN,NaN,NaN,NaN
7,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2021/hyundai/santa-fe/9507,9507,6795A,...,NaN,NaN,18990.0,142255.0,Automatic,AWD,NaN,NaN,NaN,NaN
8,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2023/hyundai/venue/9793,9793,H1274,...,NaN,NaN,18995.0,44807.0,Automatic,NaN,Gasoline,NaN,NaN,NaN
9,hyundai_gatineau,Hyundai Gatineau,Gatineau,QC,Magnetis,2026-10-22T19:52:30,https://hyundaigatineau.ca/en/pre-owned,https://hyundaigatineau.ca/en/pre-owned/2021/hyundai/kona/9792,9792,27052A,...,NaN,NaN,19344.0,26259.0,Automatic,FWD,NaN,NaN,NaN,NaN


### 3.3 Enregistrement

- un CSV par site dans `resultats/brut/` (déjà écrit par `collecter_tous()`) ;
- un fichier de tous les véhicules de la collecte : `resultats/vehicules_brut_AAAA-MM-JJ.csv` ;
- le journal de tests, complété avec `nb_lignes`, les pages collectées, le fichier et les remarques.

Encodage `utf-8-sig` partout, pour une lecture correcte dans Excel.

In [16]:
jour = datetime.now().strftime("%Y-%m-%d")
fichier_vehicules = config.DOSSIER_RESULTATS / f"vehicules_brut_{jour}.csv"
vehicules.to_csv(fichier_vehicules, index=False, encoding="utf-8-sig")
print("Véhicules :", fichier_vehicules.relative_to(config.RACINE))

journal = extraction.ajouter_collecte_au_journal(journal, rapports)
chemin = diagnostic.enregistrer_journal(journal)
print("Journal   :", chemin.relative_to(config.RACINE))
journal.reindex(columns=["nom", "nb_lignes", "pages_collectees", "problemes_remarques"])

Véhicules : resultats\vehicules_brut_2026-10-22.csv
Journal   : resultats\journal_tests.csv


,nom,nb_lignes,pages_collectees,problemes_remarques
0,Hyundai Gatineau,34,2,
1,Gatineau Acura,29,1,
2,Gatineau Kia,47,3,
3,Dupont & Dupont Honda,36,1,"Une API interne existe peut-être, mais robots.txt interdit son chemin : ne pas l'utiliser."
4,Hull Hyundai,36,1,"Une API interne existe peut-être, mais robots.txt interdit son chemin : ne pas l'utiliser."
5,Hull Nissan,12,1,"Une API interne existe peut-être, mais robots.txt interdit son chemin : ne pas l'utiliser."
6,Buckingham Chevrolet Buick GMC,36,1,"Une API interne existe peut-être, mais robots.txt interdit son chemin : ne pas l'utiliser."
7,Ottawa Honda,0,0,ARRÊT : G — accès bloqué (statut 403). Aucune tentative de contournement. || Collecte : Site non collecté (règle zér...
8,Bank Street Mazda,80,7,
9,Bank Street Hyundai,55,5,


## 4. Nettoyage et fusion (à venir)

- Harmonisation des prix entre plateformes (voir le tableau de la section 3.1).
- Normalisation des valeurs (transmission, traction, carburant, catégorie ; français et anglais).
- Suppression des doublons entre sites (inventaire en partie partagé dans le groupe Dilawri) par NIV.
- CSV final `resultats/vehicules_occasion_consolide.csv`, en UTF-8.

## 5. Documentation (à venir)

- Un court paragraphe par URL : cas détecté, technique choisie, difficultés rencontrées.
- Vérification finale : **Run All** dans un noyau neuf, sans erreur.